# 02.5 — Retrieval Evaluation: Precision, Recall, MAP, NDCG

**Problem it solves:** How do you know if your search engine is good? You need metrics that match how users experience retrieval — ranked results where the top positions matter most.

**The key insight:** Not all result positions are equal. Getting a relevant result at rank 1 is much better than at rank 10. NDCG and MAP account for this; precision/recall don't.

---

In [ ]:
import math
import numpy as np

# ---- Basic metrics ----

def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    """Of the top-k retrieved docs, what fraction are relevant?"""
    top_k = retrieved[:k]
    n_relevant = sum(1 for doc in top_k if doc in relevant)
    return n_relevant / k

def recall_at_k(retrieved: list, relevant: set, k: int) -> float:
    """Of all relevant docs, what fraction did we retrieve in top-k?"""
    top_k = retrieved[:k]
    n_relevant = sum(1 for doc in top_k if doc in relevant)
    return n_relevant / len(relevant) if relevant else 0.0

def f1_at_k(retrieved: list, relevant: set, k: int) -> float:
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

# Example
retrieved_docs = ['d1', 'd3', 'd5', 'd2', 'd8', 'd4', 'd7', 'd6', 'd9', 'd10']
relevant_docs  = {'d1', 'd2', 'd4', 'd6', 'd8'}  # 5 relevant docs exist

print("Retrieval results: ", retrieved_docs)
print("Relevant docs:     ", sorted(relevant_docs))
print()
for k in [1, 3, 5, 10]:
    p = precision_at_k(retrieved_docs, relevant_docs, k)
    r = recall_at_k(retrieved_docs, relevant_docs, k)
    f1 = f1_at_k(retrieved_docs, relevant_docs, k)
    hits = [d for d in retrieved_docs[:k] if d in relevant_docs]
    print(f"@{k:2d}: P={p:.3f}  R={r:.3f}  F1={f1:.3f}  relevant_found={hits}")

In [ ]:
# ---- Average Precision (AP) and Mean Average Precision (MAP) ----
# AP: area under the precision-recall curve, roughly
# Rewards systems that put relevant docs at the TOP of the ranking

def average_precision(retrieved: list, relevant: set) -> float:
    """
    Average Precision: average of P@k for each k where a relevant doc is found.
    A relevant doc at rank 1 contributes 1.0; at rank 10 contributes 0.1.
    """
    if not relevant:
        return 0.0
    
    precision_sum = 0.0
    n_relevant_seen = 0
    
    for k, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            n_relevant_seen += 1
            precision_sum += n_relevant_seen / k  # P@k at this hit
    
    return precision_sum / len(relevant)

def mean_average_precision(results_list: list[tuple]) -> float:
    """MAP: mean of AP over multiple queries."""
    return sum(average_precision(ret, rel) for ret, rel in results_list) / len(results_list)

# Compare two systems
system_a = ['d1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8']  # relevant ones up front
system_b = ['d3', 'd5', 'd7', 'd1', 'd2', 'd6', 'd4', 'd8']  # relevant ones spread out
relevant  = {'d1', 'd2', 'd4'}

ap_a = average_precision(system_a, relevant)
ap_b = average_precision(system_b, relevant)

print(f"System A: {system_a}")
print(f"  AP = {ap_a:.4f}")
print(f"System B: {system_b}")
print(f"  AP = {ap_b:.4f}")
print(f"Both retrieved all 3 relevant docs, but A puts them earlier -> higher AP")

# Step-by-step AP calculation for System A
print("\nAP calculation for System A:")
n_rel = 0
for k, doc in enumerate(system_a, 1):
    if doc in relevant:
        n_rel += 1
        p_at_k = n_rel / k
        print(f"  rank {k}: {doc} (relevant) -> P@{k} = {n_rel}/{k} = {p_at_k:.3f}")
    else:
        print(f"  rank {k}: {doc} (not relevant)")
print(f"  AP = sum({[round(n/k,3) for k,d in enumerate(system_a,1) for n in [sum(1 for j,x in enumerate(system_a[:k],1) if x in relevant)] if d in relevant]}) / {len(relevant)} = {ap_a:.4f}")

In [ ]:
# ---- NDCG (Normalized Discounted Cumulative Gain) ----
# Extends AP to GRADED relevance (not just relevant/not-relevant)
# A document can be 'highly relevant' (rel=3), 'relevant' (rel=1), or 'not relevant' (rel=0)

def dcg_at_k(relevances: list[float], k: int) -> float:
    """
    Discounted Cumulative Gain @ k.
    Position discount: gain at rank i divided by log2(i+1).
    Rank 1 has no discount (log2(2)=1). Rank 7: divided by log2(8)=3.
    """
    result = 0.0
    for i, rel in enumerate(relevances[:k], start=1):
        result += (2**rel - 1) / math.log2(i + 1)  # exponential gain
    return result

def ndcg_at_k(retrieved_relevances: list[float], k: int) -> float:
    """
    NDCG @ k: DCG normalized by ideal DCG (best possible ranking).
    Score in [0, 1] where 1 = perfect ranking.
    """
    actual_dcg = dcg_at_k(retrieved_relevances, k)
    ideal_dcg  = dcg_at_k(sorted(retrieved_relevances, reverse=True), k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

# Example: graded relevance (0=not rel, 1=partial, 2=relevant, 3=highly relevant)
perfect_ranking = [3, 2, 2, 1, 1, 0, 0, 0, 0, 0]  # highly relevant first
mediocre_ranking = [1, 0, 2, 0, 3, 1, 0, 2, 0, 0]  # scattered
bad_ranking      = [0, 0, 0, 1, 1, 2, 2, 3, 0, 0]  # relevant docs buried

print("NDCG at different k values:")
print(f"{'Ranking':25}  {'@1':>6}  {'@3':>6}  {'@5':>6}  {'@10':>6}")
for name, ranking in [('perfect', perfect_ranking), ('mediocre', mediocre_ranking), ('bad', bad_ranking)]:
    scores = [ndcg_at_k(ranking, k) for k in [1, 3, 5, 10]]
    print(f"{name:25}  {'  '.join(f'{s:.4f}' for s in scores)}")

print()
print("NDCG@3 vs NDCG@10: a good NDCG@3 means relevant docs are at the TOP")
print("This is what matters for users who only look at the first page")

In [ ]:
# ---- MRR (Mean Reciprocal Rank) ----
# When there's ONE correct answer (QA, single-correct-doc retrieval)

def reciprocal_rank(retrieved: list, relevant: set) -> float:
    for i, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / i
    return 0.0

def mrr(query_results: list[tuple]) -> float:
    return sum(reciprocal_rank(ret, rel) for ret, rel in query_results) / len(query_results)

# Three queries, one correct answer each
query_results = [
    (['d3', 'd1', 'd5'], {'d1'}),   # correct at rank 2 -> RR=0.5
    (['d2', 'd4', 'd2'], {'d2'}),   # correct at rank 1 -> RR=1.0
    (['d5', 'd3', 'd7'], {'d1'}),   # not found -> RR=0.0
]

for ret, rel in query_results:
    rr = reciprocal_rank(ret, rel)
    rank = next((i for i,d in enumerate(ret,1) if d in rel), None)
    print(f"Retrieved: {ret}  Relevant: {rel}  RR={rr:.3f}  (found at rank {rank})")

print(f"\nMRR = {mrr(query_results):.4f}")

In [ ]:
# ---- End-to-end evaluation of a BM25 system ----

import re, math
from collections import Counter

# Simulate a small IR benchmark
document_corpus = [
    (0, "python programming language data science"),
    (1, "java object oriented programming enterprise"),
    (2, "machine learning neural networks deep learning"),
    (3, "natural language processing text classification"),
    (4, "information retrieval search engines ranking"),
    (5, "python web framework django flask"),
    (6, "deep neural networks image recognition computer vision"),
    (7, "sql database query optimization"),
    (8, "nlp transformers bert language models"),
    (9, "search ranking algorithms relevance scoring"),
]

# Gold standard: query -> set of relevant doc IDs
qrels = {
    'python programming':   {0, 5},
    'deep learning':        {2, 6},
    'search ranking':       {4, 9},
    'nlp transformers':     {3, 8},
}

# Build BM25 (simplified inline)
def tokenize(t): return re.findall(r'\b[a-z]+\b', t.lower())

texts = [text for _, text in document_corpus]
doc_freqs = [Counter(tokenize(t)) for t in texts]
doc_lens = [len(tokenize(t)) for t in texts]
avgdl = sum(doc_lens) / len(doc_lens)
N = len(texts)
df = Counter()
for t in texts: df.update(set(tokenize(t)))
idf = {t: math.log((N-f+0.5)/(f+0.5)+1) for t,f in df.items()}

def bm25_score(query, doc_id, k1=1.5, b=0.75):
    s=0
    dl=doc_lens[doc_id]
    for t in tokenize(query):
        if t not in idf: continue
        tf=doc_freqs[doc_id].get(t,0)
        s+=idf[t]*tf*(k1+1)/(tf+k1*(1-b+b*dl/avgdl))
    return s

def retrieve(query, top_k=10):
    scores=sorted(range(N), key=lambda i: bm25_score(query,i), reverse=True)
    return scores[:top_k]

# Evaluate across all queries
print(f"{'Query':30} {'AP':>8} {'NDCG@5':>8} {'R@5':>8}")
print('-' * 60)
aps, ndcgs, recalls = [], [], []
for query, relevant in qrels.items():
    retrieved = retrieve(query, top_k=10)
    rel_grades = [2 if d in relevant else 0 for d in retrieved]
    ap = average_precision(retrieved, relevant)
    ndcg = ndcg_at_k(rel_grades, 5)
    rec = recall_at_k(retrieved, relevant, 5)
    aps.append(ap); ndcgs.append(ndcg); recalls.append(rec)
    print(f"{query:30} {ap:8.4f} {ndcg:8.4f} {rec:8.4f}")

print('-' * 60)
print(f"{'MAP / Mean NDCG@5 / Mean R@5':30} {sum(aps)/len(aps):8.4f} {sum(ndcgs)/len(ndcgs):8.4f} {sum(recalls)/len(recalls):8.4f}")

## Summary

| Metric | Measures | When to use |
|--------|----------|-------------|
| P@k | Top-k precision | When users look at first k results |
| Recall@k | Coverage of relevant docs | When missing results is costly |
| MAP | Overall ranking quality | Standard IR benchmark metric |
| NDCG@k | Graded relevance, position-weighted | When results have graded relevance |
| MRR | Position of first correct answer | QA, single-answer retrieval |

**What these metrics miss:**
- Diversity: 10 very similar relevant docs isn't as good as 10 diverse ones
- Freshness: a 2020 result might be less useful than a 2024 result
- User intent: navigational vs informational vs transactional queries need different metrics
- Actual user behavior (clicks, dwell time) is better than offline metrics

---
**Module 02 complete. Next: Module 03 — Classical ML for Text, starting with Naive Bayes**